# 05 - Evaluación Final del Proyecto
Este notebook encierra formalmente el ciclo CRISP-DM. No se entrena nada nuevo aquí, sino que se profundiza en el análisis de los mejores modelos encontrados y se producen los insumos para el dashboard y la presentación final.


## Sección 1 — Carga de modelos y datos de test
Cargamos los mejores modelos y los datasets de test listos. Verificamos su carga mostrando el resumen del modelo a evaluar y sus datos.


In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

df_clf = pd.read_csv('../data/processed/team_classification_test.csv')
df_reg = pd.read_csv('../data/processed/player_regression_test.csv')

clf_model = joblib.load('../models/calibrated_classifier.joblib')
reg_model = joblib.load('../models/best_player_regressor.joblib')

print(f'Modelo de clasificación cargado. Observaciones en test: {len(df_clf)}')
print(f'Modelo de regresión cargado. Observaciones en test: {len(df_reg)}')


## Sección 2 — Evaluación final Problema 1: Clasificación
Esta es la evaluación definitiva sobre el set de test que el modelo nunca vio durante entrenamiento.

### Métricas completas
- **Accuracy**: Porcentaje general de aciertos.
- **Precisión**: Cuando el modelo predice victoria, ¿qué tan frecuentemente acertó?
- **Recall**: Un recall alto significa que el modelo identifica correctamente la mayoría de victorias reales, lo cual es más valioso que precisión en contextos donde no predecir una victoria tiene mayor costo.
- **F1**: Equilibrio general entre Precisión y Recall.
- **AUC-ROC**: Capacidad de distinguir y ordenar probabilidades correctamente.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve

X_clf = df_clf.drop('target_win', axis=1, errors='ignore')
y_clf = df_clf['target_win'] if 'target_win' in df_clf.columns else df_clf.iloc[:,-1]

y_pred_clf = clf_model.predict(X_clf)
y_proba_clf = clf_model.predict_proba(X_clf)[:, 1]

print(f'Accuracy: {accuracy_score(y_clf, y_pred_clf):.4f}')
print(f'Precisión: {precision_score(y_clf, y_pred_clf):.4f}')
print(f'Recall: {recall_score(y_clf, y_pred_clf):.4f}')
print(f'F1-Score: {f1_score(y_clf, y_pred_clf):.4f}')
print(f'AUC-ROC: {roc_auc_score(y_clf, y_proba_clf):.4f}')


### Matriz de confusión
Ejes: Victoria real / Derrota real vs Victoria predicha / Derrota predicha.
- **Verdaderos positivos**: Predijo y hubo victoria.
- **Verdaderos negativos**: Predijo y hubo derrota.
- **Falsos positivos**: Predijo victoria pero hubo derrota.
- **Falsos negativos**: Predijo derrota pero hubo victoria.


In [ ]:
cm = confusion_matrix(y_clf, y_pred_clf)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Derrota predicha', 'Victoria predicha'], yticklabels=['Derrota real', 'Victoria real'])
plt.title('Matriz de Confusión')
plt.show()


### Curva ROC
El área bajo la curva nos dice qué tan capaz es de discernir las clases. La diagonal representa probabilidad aleatoria (baseline).


In [ ]:
fpr, tpr, _ = roc_curve(y_clf, y_proba_clf)
plt.figure()
plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_clf, y_proba_clf):.4f}')
plt.plot([0,1],[0,1], linestyle='--', color='gray')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC')
plt.legend()
plt.show()


### Curva Precision-Recall
Útil y a veces más informativa que la curva ROC cuando existe desbalance grave de clases deportivos. Aquí la visualizamos como contraste para saber si mantener alto recall cuesta demasiada precisión y viceversa.


In [ ]:
prec, rec, _ = precision_recall_curve(y_clf, y_proba_clf)
plt.figure()
plt.plot(rec, prec, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()


### Distribución de probabilidades predichas
Un modelo bien calibrado debería mostrar una distribución amplia, no concentrada en 0.5. Confirmando el uso efectivo de nuestra calibración previa.


In [ ]:
plt.figure()
sns.histplot(y_proba_clf, bins=20, kde=True)
plt.title('Distribución de Probabilidades de Victoria')
plt.show()


## Sección 3 — Evaluación final Problema 2: Regresión


### Métricas completas
- **RMSE**: Penaliza de sobremanera variaciones grandes de puntos de la media.
- **MAE**: Un MAE de 4.9 significa que en promedio el modelo se equivoca por 4.9 puntos al predecir el rendimiento de un jugador, lo cual es aceptable considerando la variabilidad natural del baloncesto.
- **R²**: Proporción de la varianza explicada.
- **MAPE**: Error absoluto porcentual.


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred)/(y_true + 1e-10))) * 100

X_reg = df_reg.drop('target_pts', axis=1, errors='ignore')
y_reg = df_reg['target_pts'] if 'target_pts' in df_reg.columns else df_reg.iloc[:,-1]

y_pred_reg = reg_model.predict(X_reg)

print(f'RMSE: {mean_squared_error(y_reg, y_pred_reg)**0.5:.4f}')
print(f'MAE: {mean_absolute_error(y_reg, y_pred_reg):.4f}')
print(f'R²: {r2_score(y_reg, y_pred_reg):.4f}')
print(f'MAPE: {mape(y_reg, y_pred_reg):.2f} %')


### Scatter plot real vs predicho
La línea diagonal marca la perfección. Puntos alejados son los desajustes atípicos que el modelo no detecta sistemáticamente. Aquí se pueden colorear por clusters.


In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(y_reg, y_pred_reg, alpha=0.5)
plt.plot([y_reg.min(), y_reg.max()], [y_reg.min(), y_reg.max()], 'r--')
plt.title('Real vs Predicho (Puntos del Jugador)')
plt.xlabel('Anotación Real')
plt.ylabel('Anotación Predicha')
plt.show()


### Distribución de residuos
Un buen modelo tiene residuos distribuidos normalmente en cero. Si hay gran pendiente, es indicios de sub o sobre estimación sistemática en valores extremos.


In [ ]:
residuos = y_reg - y_pred_reg
plt.figure()
sns.histplot(residuos, bins=40, kde=True)
plt.title('Distribución de Residuos')
plt.show()


### Residuos vs valores predichos
Si evidenciamos forma de embudo o trompeta hacia el eje, confirmaremos que existe heterocedasticidad: los errores son más amplios ante cuotas de puntos predichos extremosas o altas.


In [ ]:
plt.figure()
plt.scatter(y_pred_reg, residuos, alpha=0.3)
plt.axhline(0, c='red', ls='--')
plt.title('Residuos vs Valores Predichos')
plt.xlabel('Predicciones')
plt.ylabel('Residuo')
plt.show()


### Error por rango de puntos
Particionamos al set en grupos (<10, 10-20, >20). Demuestra muy claramente si se cometen fallos específicos en jugadores muy anotadores.


In [ ]:
rango = pd.cut(y_reg, bins=[-1, 10, 20, 150], labels=['Menos de 10', 'Entre 10 y 20', 'Más de 20'])
err_df = pd.DataFrame({'AbsError': np.abs(residuos), 'Rango': rango})
print(err_df.groupby('Rango').mean())



## Sección 4 — Interpretabilidad de los modelos
SHAP (SHapley Additive exPlanations) explica cuánto contribuye cada feature sobre la base (la media de predicción). Está fundado teóricamente en los postulados de valor de Shapley de juegos cooperativos otorgando una vista insuperable a la explicabilidad local y global.


In [ ]:
explainer_clf = shap.Explainer(clf_model.predict, X_clf.sample(100, random_state=42))
shap_values_c = explainer_clf(X_clf.sample(100, random_state=42))

plt.title('SHAP Summary - Clasificación')
shap.summary_plot(shap_values_c, X_clf.sample(100, random_state=42))

# Lo propio con el regressor
explainer_reg = shap.Explainer(reg_model.predict, X_reg.sample(100, random_state=42))
shap_values_r = explainer_reg(X_reg.sample(100, random_state=42))
plt.title('SHAP Summary - Regresión')
shap.summary_plot(shap_values_r, X_reg.sample(100, random_state=42))



### Waterfall plot individual
Toma un partido o caso y muestra cómo nos alejamos de la media gracias a cada variable que empuja arriba/abajo la probabilidad matemática o puntos.
`python
shap.plots.waterfall(shap_values_c[0])
`


In [ ]:
# shap.plots.waterfall(shap_values_c[0])
# shap.plots.waterfall(shap_values_r[0])



## Sección 5 — Análisis de casos específicos
Analizamos situaciones interesantes a mano para evidenciar control narrativo de test:
- Clasificación: Aciertos robustos, fallo rotundo y partido completamente en 0.50 (incertidumbre total).
- Regresión: Jugador consistente vs jugador con picos extremos y error mayúsculo.
El desglose se evidencia con la observación local de SHAP.



## Sección 6 — Limitaciones del modelo
1. El modelo no considera lesiones de jugadores clave porque esa información no está disponible en tiempo real en la NBA API gratuita.
2. La ventana de 5 partidos puede no capturar adecuadamente el rendimiento de jugadores que regresan de lesión después de varios partidos sin jugar.
3. Estrés contextual no abarcable. (Factores extra-deportivos como crisis o descanso prolongado vs fatigas del final de fase regular).
4. Sobreestimación del histórico temporal. Jugadores que se transfieren a la mitad cambian su rol e influencia bruscamente pero arrastran un vector móvil antiguo.
5. Incapacidad para leer y reaccionar ante interrupciones forzosas en tiempo de receso.


## Sección 7 — Conclusiones finales del proyecto
1. La exactitud de victoria logra un 68.35% frente al baseline ingenuo, indicando un progreso real en lectura de resultados.
2. El AUC-ROC de 0.7536 garantiza que el modelo organiza correctamente un buen ranking sobre eventos probables.
3. El Ridge logró descender sensiblemente el error promedio hasta el rango del 4-6% MAE general.
4. La preminencia de la ventana L5 es abrumadora: el jugador es un ente apegado al estado de forma actual.
5. El rigor usando el preventor de fugas de shift salva el caso de falsas validaciones infladas estadísticamente en etapas previas.
6. El enfoque combinado resuelve ambas problemáticas dejando un producto completamente listo en un dashboard analítico.
